# Tool Calling

This notebook demonstrates tool invocation using the inference adaptor and the tool runtime adaptors with the SambaNova cloud-based instruct models.

Tools are functions that can be invoked by an agent to perform tasks.

They are organized into tool groups and registered with specific providers. Each tool group represents a collection of related tools from a single provider. They are organized into groups so that state can be externalized, so that the collection operates on the same state. An example of this would be a `db_access` tool group that contains tools for interacting with a database. `list_tables`, `query_table`, `insert_row` could be examples of tools in this group.

Tools are treated as any other resource in OGX like models. You can register them, have providers for them etc.

When instantiating an agent, you can provide it a list of tool groups that it has access to.
Agent gets the corresponding tool definitions for the specified tool groups and passes them along to the model.

Please refer to the [OGX documentation on tool calling](https://ogx-ai.github.io/docs/building_applications/tools) for further details.

## Setup

In [ ]:
# Imports
import os
from ogx_client import OgxClient, Agent, AgentEventLogger

# Select model
model = "sambanova/sambanova/Meta-Llama-3.3-70B-Instruct"


In [3]:
# Create HTTP client
OGX_PORT = 8321
client = OgxClient(base_url=f"http://localhost:{OGX_PORT}")


## Types of Tool Group providers
There are three types of providers for tool groups that are supported by Llama Stack.

1. Built-in providers.
2. Model Context Protocol (MCP) providers.
3. Client provided tools.

## Built-in providers
Built-in providers come packaged with Llama Stack. These providers provide common functionalities like web search, code interpretation, and computational capabilities.

### Web Search providers
There are three web search providers that are supported by Llama Stack.

- Brave Search.
- Bing Search.
- Tavily Search.

### WolframAlpha
The WolframAlpha tool provides access to computational knowledge through the WolframAlpha API.



## List available tool groups on the provider

> **Note:** Server-side tool groups (web search, WolframAlpha, MCP) are configured in
> the distribution config. Use `client.routes.list()` to inspect available API routes.


## Tool invocation via Agent
Server-side tools are invoked automatically by the Agent when it decides to use them.
Simply configure the tool in the agent's `tools` list and the agent handles calling it:

```python
agent = Agent(client, model=model, instructions="...", tools=[{"type": "web_search"}])
response = agent.create_turn(messages=[{"role": "user", "content": "Search for..."}], session_id=session_id)
```


## Custom Tools

When you want to use tools other than the built-in tools, you just need to implement a python function with a docstring. The content of the docstring will be used to describe the tool and the parameters and passed along to the generative model.

NOTE: You should employ python docstrings to describe the tool and the parameters. It is important to document the tool and the parameters so that the model can use the tool correctly. It is recommended to experiment with different docstrings to see how they affect the model’s behavior.

In [5]:
def check_prime(n: int) -> bool:
    """
    Determines whether a given integer is a prime number.

    :param n: The integer to check.
    :return: True if 'n' is prime, otherwise False.
    """
    if n < 2:
        return False
    for i in range(2, int(n ** 0.5) + 1):
        if n % i == 0:
            return False
    return True

## Agent with tools
Once defined, simply pass the tool to the agent config. Agent will take care of the rest (calling the model with the tool definition, executing the tool, and returning the result to the model for the next iteration).

In [6]:
# Example agent config with a client-provided Python tool
agent = Agent(
    client,
    model=model,
    instructions="You are a helpful assistant",
    tools=[check_prime],
)


In [7]:
session_id = agent.create_session("custom-tool-session")

response = agent.create_turn(
    messages=[{"role": "user", "content": "Is 25 a prime number?"}],
    session_id=session_id,
    stream=True,
)
for log in AgentEventLogger().log(response):
    print(log, end="")
